# 08 - Hyperparameter Tuning

## Overview

Hyperparameter tuning for XGBoost and LightGBM using RandomizedSearchCV.

**Note**: This notebook uses a smaller sample for faster tuning. Full tuning can take hours.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import SEED

import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score

import xgboost as xgb
import lightgbm as lgb

# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
MODELS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/models"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"

os.makedirs(MODELS_DIR, exist_ok=True)

print("Hyperparameter tuning notebook initialized.")

## 1. Load Data (Subsampled for Faster Tuning)

In [ ]:
print("Loading data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

# Use 15% of data for tuning (faster)
np.random.seed(SEED)
sample_size = int(len(train_df) * 0.15)
sample_idx = np.random.choice(len(train_df), size=sample_size, replace=False)
train_sample = train_df.iloc[sample_idx]

X = train_sample[feature_cols].values
y = train_sample['label'].values

print(f"Tuning on {sample_size:,} samples ({sample_size/len(train_df)*100:.1f}% of data)")
print(f"Features: {len(feature_cols)}, Classes: {num_classes}")

## 2. XGBoost Hyperparameter Grid

In [ ]:
# XGBoost parameter grid
xgb_param_grid = {
    'max_depth': [6, 8, 10, 12],
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5]
}

print("XGBoost parameter grid:")
for k, v in xgb_param_grid.items():
    print(f"  {k}: {v}")

In [ ]:
# XGBoost RandomizedSearchCV
print("\n" + "=" * 60)
print("XGBOOST HYPERPARAMETER TUNING")
print("=" * 60)

# Base model
xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    tree_method='hist',
    random_state=SEED,
    verbosity=0,
    n_jobs=8
)

# Cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# RandomizedSearch
start_time = time.time()

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=20,  # Number of random combinations to try
    scoring='f1_macro',
    cv=cv,
    verbose=2,
    random_state=SEED,
    n_jobs=1  # Use 1 to avoid memory issues
)

xgb_search.fit(X, y)

xgb_time = time.time() - start_time
print(f"\nXGBoost tuning completed in {xgb_time:.1f}s ({xgb_time/60:.1f} min)")

In [ ]:
# Best XGBoost parameters
print("\nBest XGBoost Parameters:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV F1 Macro: {xgb_search.best_score_:.4f}")

# Save best params
best_xgb_params = xgb_search.best_params_.copy()
best_xgb_params['objective'] = 'multi:softprob'
best_xgb_params['num_class'] = num_classes
best_xgb_params['tree_method'] = 'hist'
best_xgb_params['random_state'] = SEED
best_xgb_params['verbosity'] = 0
best_xgb_params['n_jobs'] = 8

## 3. LightGBM Hyperparameter Grid

In [ ]:
# LightGBM parameter grid
lgb_param_grid = {
    'max_depth': [6, 8, 10, 12],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_samples': [10, 20, 50],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5]
}

print("\nLightGBM parameter grid:")
for k, v in lgb_param_grid.items():
    print(f"  {k}: {v}")

In [ ]:
# LightGBM RandomizedSearchCV
print("\n" + "=" * 60)
print("LIGHTGBM HYPERPARAMETER TUNING")
print("=" * 60)

# Base model
lgb_base = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=num_classes,
    random_state=SEED,
    verbose=-1,
    n_jobs=8
)

# RandomizedSearch
start_time = time.time()

lgb_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_grid,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    verbose=2,
    random_state=SEED,
    n_jobs=1
)

lgb_search.fit(X, y)

lgb_time = time.time() - start_time
print(f"\nLightGBM tuning completed in {lgb_time:.1f}s ({lgb_time/60:.1f} min)")

In [ ]:
# Best LightGBM parameters
print("\nBest LightGBM Parameters:")
for k, v in lgb_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV F1 Macro: {lgb_search.best_score_:.4f}")

# Save best params
best_lgb_params = lgb_search.best_params_.copy()
best_lgb_params['objective'] = 'multiclass'
best_lgb_params['num_class'] = num_classes
best_lgb_params['random_state'] = SEED
best_lgb_params['verbose'] = -1
best_lgb_params['n_jobs'] = 8

## 4. Comparison of Tuned Models

In [ ]:
# Train final models with best params
print("\n" + "=" * 60)
print("TRAINING FINAL MODELS WITH BEST PARAMS")
print("=" * 60)

# Load validation data
val_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "val.parquet"))
X_val = val_df[feature_cols].values
y_val = val_df['label'].values

# XGBoost
print("\nTraining XGBoost with best params...")
xgb_final = xgb.XGBClassifier(**best_xgb_params)
xgb_final.fit(X, y, eval_set=[(X_val, y_val)], verbose=False)
xgb_pred = xgb_final.predict(X_val)
xgb_f1 = f1_score(y_val, xgb_pred, average='macro')
print(f"XGBoost Validation F1 Macro: {xgb_f1:.4f}")

# LightGBM
print("\nTraining LightGBM with best params...")
lgb_final = lgb.LGBMClassifier(**best_lgb_params)
lgb_final.fit(X, y, eval_set=[(X_val, y_val)])
lgb_pred = lgb_final.predict(X_val)
lgb_f1 = f1_score(y_val, lgb_pred, average='macro')
print(f"LightGBM Validation F1 Macro: {lgb_f1:.4f}")

## 5. Save Tuned Parameters

In [ ]:
# Save results
tuning_results = {
    'xgboost': {
        'best_params': best_xgb_params,
        'best_cv_score': float(xgb_search.best_score_),
        'tuning_time': float(xgb_time),
        'validation_f1': float(xgb_f1)
    },
    'lightgbm': {
        'best_params': best_lgb_params,
        'best_cv_score': float(lgb_search.best_score_),
        'tuning_time': float(lgb_time),
        'validation_f1': float(lgb_f1)
    },
    'config': {
        'sample_ratio': 0.15,
        'n_iter': 20,
        'cv_folds': 3
    }
}

with open(os.path.join(LOGS_DIR, '08_hyperparameter_tuning.json'), 'w') as f:
    json.dump(tuning_results, f, indent=2)

print("\nTuning results saved: results/logs/08_hyperparameter_tuning.json")

## 6. Update Config File

In [ ]:
# Write tuned params to src/config.py
import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
import re

config_path = '/home/willtanoe/Documents/fl-xgb-thesis/src/config.py'
with open(config_path, 'r') as f:
    content = f.read()

# Build new XGB_PARAMS block
xgb_lines = ['XGB_PARAMS = {\n']
for k, v in best_xgb_params.items():
    xgb_lines.append(f"    '{k}': {repr(v)},\n")
xgb_lines.append('}\n')
xgb_block = ''.join(xgb_lines)

# Build new LGB_PARAMS block
lgb_lines = ['LGB_PARAMS = {\n']
for k, v in best_lgb_params.items():
    lgb_lines.append(f"    '{k}': {repr(v)},\n")
lgb_lines.append('}\n')
lgb_block = ''.join(lgb_lines)

# Replace in config content
content = re.sub(
    r'XGB_PARAMS = \{[^}]+\}',
    xgb_block.rstrip(),
    content
)
content = re.sub(
    r'LGB_PARAMS = \{[^}]+\}',
    lgb_block.rstrip(),
    content
)

with open(config_path, 'w') as f:
    f.write(content)

print(f"Config updated: {config_path}")
print("XGB_PARAMS and LGB_PARAMS replaced with tuned values.")

In [ ]:
print("## Summary")
print()
print(f"- XGBoost best CV F1: {xgb_search.best_score_:.4f}")
print(f"- LightGBM best CV F1: {lgb_search.best_score_:.4f}")
print()